# 🏡 Greenhouse Automation — Nostradamus IoTO SDK

Greenhouse climate control with actuator feedback: air temperature, humidity, CO2, PAR light, substrate moisture/EC/pH, VPD and the state of the vent, heating, irrigation and fogger actuators.

This notebook mirrors `examples/agriculture/greenhouse_automation.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **3 zones × 48 half-hourly readings** (144 records) to the collection
`greenhouse_sensors`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `greenhouse_sensors` collection with a schema
2. Generate and send 3 zones × 48 half-hourly readings
3. Query heat stress, low CO2, active irrigation and VPD ranking
4. Aggregate air temperature, CO2 and substrate moisture statistics
5. Delete a 12-hour window for `GH_01_ZONE_A`
6. Delete the collection

## What this notebook highlights

- Heat stress detection (air temp > 35 °C with vents barely open)
- CO2 depletion detection (below 400 ppm)
- Actuator auditing (vent / heating / irrigation / fogger flags)

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🏡 Nostradamus IoTO SDK — Greenhouse Automation")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the greenhouse zone collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/agriculture/greenhouse_automation.py`.

In [ ]:
COLLECTION_NAME = "greenhouse_sensors"

COLLECTION_SCHEMA = {
                    "key": "GH_01_ZONE_A",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "air_temperature_c": 28.5,
                    "relative_humidity_pct": 65.0,
                    "co2_ppm": 800.0,
                    "par_light_umol": 450.0,
                    "substrate_moisture_pct": 55.0,
                    "substrate_ec_ms": 2.1,
                    "substrate_ph": 6.2,
                    "vpd_kpa": 1.2,
                    "vent_position_pct": 40.0,
                    "heating_active": 0,
                    "irrigation_active": 1,
                    "fogger_active": 0,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Greenhouse climate and actuator data",
            tags=["greenhouse", "climate", "automation"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send greenhouse climate data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_greenhouse_data()` is the
same generator used by the script.

In [ ]:
def generate_greenhouse_data(num_zones=3, num_readings=48):
    """Generate greenhouse climate and actuator data.

    Args:
        num_zones: Number of greenhouse zones
        num_readings: Readings per zone (30-min intervals over 24h)

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)
    zone_ids = ["GH_01_ZONE_A", "GH_01_ZONE_B", "GH_02_ZONE_A"]

    for zone_idx in range(min(num_zones, len(zone_ids))):
        zone_id = zone_ids[zone_idx]
        temp_base = 22.0 + zone_idx * 2.0

        for i in range(num_readings):
            timestamp = base_time + timedelta(minutes=30 * i)
            hour = timestamp.hour

            is_daytime = 6 <= hour <= 20
            solar_factor = max(0, 1.0 - abs(hour - 13) / 7.0)

            air_temp = temp_base + solar_factor * 12.0
            air_temp += random.uniform(-2, 2)
            air_temp = round(min(42, max(18, air_temp)), 1)

            humidity = 70.0 - solar_factor * 20.0
            humidity += random.uniform(-5, 5)
            humidity = round(min(95, max(40, humidity)), 1)

            co2 = 400 + random.uniform(-50, 600)
            if is_daytime:
                co2 -= 100 * solar_factor
            co2 = round(min(1200, max(350, co2)), 0)

            par = solar_factor * 600 + random.uniform(-50, 100)
            par = round(max(0, min(800, par)), 0)

            moisture = 50.0 + random.uniform(-20, 15)
            ec = round(random.uniform(0.5, 4.0), 2)
            ph = round(random.uniform(5.5, 7.0), 1)

            vpd = round(random.uniform(0.4, 2.5), 2)

            vent = 0.0
            if air_temp > 28:
                vent = min(100, (air_temp - 28) * 15)
            vent = round(vent + random.uniform(-5, 5), 0)
            vent = max(0, min(100, vent))

            heating = 1 if air_temp < 20 else 0
            irrigation = 1 if moisture < 35 else 0
            fogger = 1 if humidity < 50 and is_daytime else 0

            reading = {
                "key": zone_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "air_temperature_c": air_temp,
                "relative_humidity_pct": humidity,
                "co2_ppm": co2,
                "par_light_umol": par,
                "substrate_moisture_pct": round(moisture, 1),
                "substrate_ec_ms": ec,
                "substrate_ph": ph,
                "vpd_kpa": vpd,
                "vent_position_pct": vent,
                "heating_active": heating,
                "irrigation_active": irrigation,
                "fogger_active": fogger,
            }
            data.append(reading)

    return data

data = generate_greenhouse_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🔥 Heat stress (air temp > 35 °C and vents < 50 %)**
- **🫁 Low CO2 (< 400 ppm)**
- **💧 Irrigation active**
- **Top 5 highest VPD**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🔥 Heat stress (air temp > 35 °C and vents < 50 %)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "air_temperature_c", "operator": "gt", "property_value": 35},
            {"property_name": "vent_position_pct", "operator": "lt", "property_value": 50},
        ],
    )
    summary(result, "🔥 Heat stress (air temp > 35 °C and vents < 50 %)")

    # 🫁 Low CO2 (< 400 ppm)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "co2_ppm", "operator": "lt", "property_value": 400},
        ],
    )
    summary(result, "🫁 Low CO2 (< 400 ppm)")

    # 💧 Irrigation active
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "irrigation_active", "operator": "eq", "property_value": 1},
        ],
    )
    summary(result, "💧 Irrigation active")

    # Top 5 highest VPD
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "vpd_kpa", "air_temperature_c"],
        order_by='{"field": "vpd_kpa", "order": "desc"}',
        limit=5,
    )
    print("🏆 Top 5 highest VPD")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average air temperature (°C)", "air_temperature_c", StatOperation.AVG),
        ("Maximum CO2 (ppm)", "co2_ppm", StatOperation.MAX),
        ("Minimum substrate moisture (%)", "substrate_moisture_pct", StatOperation.MIN),
        ("Distinct zones", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one greenhouse zone

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`GH_01_ZONE_A` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "GH_01_ZONE_A"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")